# Diachronic Language Analysis using Word Embeddings

This notebook documents the **complete pipeline for diachronic linguistic analysis** using CBOW word embeddings trained on Google Books N-grams data.

The analysis is divided into **8 main phases**:

1. **Configuration** - Setup of global parameters
2. **Preprocessing** - Cleaning and tokenization of raw data
3. **Vocabulary Building** - Creation of a shared vocabulary
4. **PyTorch Dataset** - Data preparation for training
5. **Training** - Training of CBOW models for each decade
6. **Alignment** - Semantic space alignment using Procrustes
7. **Semantic Drift** - Analysis of semantic word evolution
8. **Visualization** - PCA and t-SNE representation


### What is Semantic Drift?

**Semantic drift** is the change in meaning of a word over time. For example:

- **"computer"** in the 1930s referred to a person performing manual calculations
- **"computer"** in the 1990s refers to an electronic device

By analyzing changes in embedding vectors over time, we can quantify and visualize these semantic shifts.


### Why Google Books N-grams?

The **Google Books N-grams v3** dataset provides:
- **100+ million digitized books**
- **Temporal coverage** from 1900 to 2019
- **Accurate frequencies** for every word in every year
- **Representativeness** of contemporary literary language

This dataset is ideal for historical language studies since it reflects the real usage of words over time.

### Setup & Configuration

In [1]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import glob

# Setup path to project root
PROJECT_ROOT = Path('/home/ccoppola/projects/diachronic_text_analysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/ccoppola/projects/diachronic_text_analysis


In [2]:
# Import configuration module
import importlib
import src.config
importlib.reload(src.config)

from src.config import (
    DATA_RAW_EXPANDED_DIR,
    DATA_PROCESSED_EXPANDED_DIR,
    get_preprocess_input_file,
    get_vocab_path,
    get_decade_from_year,
    NGRAM_TYPE,
    VOCAB_SIZE,
    EMBEDDING_DIM,
    CONTEXT_WINDOW,
    START_YEAR,
    END_YEAR
)

print("Configuration imported and reloaded successfully")

Configuration imported and reloaded successfully


The project is configured to work with the 5gram-expanded variant using the official dataset paths from `config.py`.

---

## Phase 1: Download Dataset

The **Google Books N-grams v3** dataset provides word frequencies extracted from 100+ million digitized books (1900-2019).

**Raw data format (TSV - Tab Separated Values):**
```
ngram [TAB] year [TAB] match_count [TAB] volume_count
```

- **ngram**: sequence of N words (e.g., "the quick brown" for 5-grams)
- **year**: publication year (1900-2019)
- **match_count**: occurrences of this n-gram in corpus for that year
- **volume_count**: number of different books containing this n-gram

This raw data is the starting point - it will be cleaned and aggregated in later pipeline phases.

In [14]:
# Read a sample of real data from the raw file
raw_file = Path(get_preprocess_input_file(expanded=True))

sample_size = 15
sample_data = []

with open(raw_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            sample_data.append({
                'year': int(parts[1]),
                'ngram': parts[0],
                'match_count': int(parts[2])
            })

df_raw_sample = pd.DataFrame(sample_data)

# Display with pandas formatting and hide index
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', None)

styled_table = (
    df_raw_sample[['year', 'ngram', 'match_count']]
    .style
    .hide(axis='index')
    .set_properties(**{'text-align': 'center'})
)
display(styled_table)

year,ngram,match_count
1903,account of principal and interest,2
1904,account of principal and interest,3
1905,account of principal and interest,4
1908,account of principal and interest,2
1909,account of principal and interest,1
1913,account of principal and interest,3
1914,account of principal and interest,5
1915,account of principal and interest,5
1916,account of principal and interest,2
1917,account of principal and interest,4


Each line in the raw dataset represents the frequency of a 5-gram in a specific year.

<span style="color:#ff7f0e;"><strong>Example</strong></span>: The first line shows that the 5-gram "account of principal and interest" appeared <strong>2 times</strong> (`match_count`) in 1903 (which maps to the 1900s decade).

<span style="color:#2ca02c;"><strong>Key observations</strong></span>:

- <span style="color:#9467bd;"><strong>Repeated entries</strong></span>: each n-gram can appear multiple times in the file (for different years).
- <span style="color:#d62728;"><strong>Frequency signal</strong></span>: `match_count` quantifies how frequently each 5-gram appears in a given year.
- <span style="color:#17becf;"><strong>Temporal variation</strong></span>: the same n-gram may have varying frequencies across years, showing how language usage changes over time.
- <span style="color:#8c564b;"><strong>Next processing step</strong></span>: this raw data will be preprocessed (cleaned, tokenized) and aggregated by decade in Phase 2.
- <span style="color:#e377c2;"><strong>Training purpose</strong></span>: aggregated frequency data is used to build a shared vocabulary and train embeddings across all decades.

### About This Dataset

The observed dataset is **not the raw full Google N-grams dump**, but a curated subset produced during the **download phase with controlled filters**.

<span style="color:#1f77b4;"><strong>Why these FILE_RANGES were used</strong></span>:

- <span style="color:#2ca02c;"><strong>Computational feasibility</strong></span>: downloading and processing the entire Google Books N-grams archive is extremely expensive in time, storage, and I/O.
- <span style="color:#ff7f0e;"><strong>Representative coverage</strong></span>: selecting multiple alphabetical intervals (`[2000-2005]`, `[3000-3005]`, `[4000-4005]`, `[5000-5005]`, `[6000-6005]`, `[7000-7005]`) samples different portions of the corpus instead of overfocusing on a narrow segment.
- <span style="color:#d62728;"><strong>Stable experimentation</strong></span>: fixed ranges make the pipeline reproducible, so model training and drift analysis can be repeated consistently.

During download from Google Books N-grams v3 (2020):

- <span style="color:#9467bd;"><strong>Subset selection</strong></span>: only files in the configured `FILE_RANGES` intervals are included.
- <span style="color:#8c564b;"><strong>Frequency filter</strong></span>: `MIN_CORPUS_OCCURRENCES = 100` removes very rare n-grams (appearing fewer than 100 times in the historical corpus).
- <span style="color:#17becf;"><strong>Alphabetic filter</strong></span>: `ONLY_ALPHABETIC = True` keeps linguistically meaningful strings and removes numbers, punctuation, and symbols.

These design choices reduce noise (OCR artifacts, malformed strings, non-linguistic patterns) while preserving enough lexical variety for diachronic analysis. The output is then ready for **Phase 2 preprocessing**, where n-grams are tokenized, normalized, and aggregated by decade.

---

## Phase 2: Preprocessing Pipeline

This section demonstrates how **RAW N-Grams** are transformed into **clean**, **decade-aggregated text data** ready for **vocabulary building** and embedding training.

### Text Normalization and Tokenization

The preprocessing phase applies linguistic transformations to raw n-grams:

1. <span style="color:#1f77b4;"><strong>Normalization</strong></span>: convert to lowercase and remove Unicode accents.
2. <span style="color:#2ca02c;"><strong>Tokenization</strong></span>: split text into individual words.
3. <span style="color:#d62728;"><strong>Validation</strong></span>: filter out non-linguistic tokens (numbers, URLs, artificial repetitions).
4. <span style="color:#ff7f0e;"><strong>Number Replacement</strong></span>: replace numeric tokens with the `NUM` token to preserve syntactic patterns.
5. <span style="color:#9467bd;"><strong>Frequency Aggregation</strong></span>: accumulate `match_count` values by decade (not just occurrence counts).

In [4]:
# Preprocessing demo on real rows: compact version
import unicodedata
import re
from pathlib import Path
from IPython.display import display, Markdown, HTML

# --- Tokenization/preprocessing functions (preserved) ---
def normalize_text(text: str) -> str:
    """Normalize: lowercase + Unicode accent removal."""
    text = text.lower()
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

def is_number(token: str) -> bool:
    """Check if token is a pure number or date."""
    clean = token.replace(',', '').replace('.', '')
    return clean.isdigit()

def is_valid_token(token: str, min_len=2, max_len=40, min_alpha_ratio=0.6) -> bool:
    """Check if token is linguistically valid."""
    if len(token) < min_len or len(token) > max_len:
        return False
    if any(p in token for p in ['http', 'www', '.com', '#', '@']):
        return False
    alpha_count = sum(1 for c in token if c.isalpha())
    if alpha_count == 0:
        return False
    return (alpha_count / len(token)) >= min_alpha_ratio

def tokenize_and_clean(text: str) -> list:
    """Complete tokenization and cleaning pipeline."""
    text = normalize_text(text)
    tokens = re.findall(r'\b[\w]+\b', text)
    cleaned = []
    for token in tokens:
        if is_number(token):
            cleaned.append('NUM')
        elif is_valid_token(token):
            cleaned.append(token)
    return cleaned

# --- Compact sampling logic ---
MAX_SCAN_LINES = 1_500_000
WANTED_KEPT = 8
WANTED_DISCARDED = 8

processed_cache = {}

def in_processed_decade(decade: str, ngram: str) -> bool:
    if decade not in processed_cache:
        decade_path = Path(DATA_PROCESSED_EXPANDED_DIR) / f"{decade}.txt"
        if decade_path.exists():
            with open(decade_path, 'r', encoding='utf-8') as f:
                processed_cache[decade] = set(line.strip() for line in f if line.strip())
        else:
            processed_cache[decade] = set()
    return ngram in processed_cache[decade]

kept_rows, discarded_rows = [], []
seen_kept, seen_discarded = set(), set()
changed_rows_scanned = 0

with open(raw_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= MAX_SCAN_LINES:
            break

        parts = line.strip().split('\t')
        if len(parts) < 3:
            continue

        raw_ngram = parts[0]
        year = int(parts[1])
        decade = get_decade_from_year(year)

        normalized = normalize_text(raw_ngram)
        normalized_tokens = re.findall(r'\b[\w]+\b', normalized)
        cleaned_tokens = tokenize_and_clean(raw_ngram)

        # Keep only rows with visible preprocessing effect
        if raw_ngram == normalized and normalized_tokens == cleaned_tokens:
            continue

        changed_rows_scanned += 1

        # Discarded case (< NGRAM_TYPE tokens)
        if len(cleaned_tokens) < NGRAM_TYPE and len(discarded_rows) < WANTED_DISCARDED:
            processed_ngram = ' '.join(cleaned_tokens) if cleaned_tokens else '[filtered]'
            key = processed_ngram
            if key not in seen_discarded:
                seen_discarded.add(key)
                discarded_rows.append({
                    'Year': year,
                    'Raw N-gram': raw_ngram,
                    'Processed N-gram': processed_ngram,
                    'Outcome': 'Discarded (<5 tokens)',
                    'Result': "<span style='color:#c62828;font-weight:700;'>X</span>"
                })

        # Kept case (>= NGRAM_TYPE tokens and present in real processed files)
        elif len(cleaned_tokens) >= NGRAM_TYPE and len(kept_rows) < WANTED_KEPT:
            processed_ngram = ' '.join(cleaned_tokens[:NGRAM_TYPE])
            if in_processed_decade(decade, processed_ngram) and processed_ngram not in seen_kept:
                seen_kept.add(processed_ngram)
                kept_rows.append({
                    'Year': year,
                    'Raw N-gram': raw_ngram,
                    'Processed N-gram': processed_ngram,
                    'Outcome': 'Kept',
                    'Result': "<span style='color:#2e7d32;font-weight:700;'>OK</span>"
                })

        if len(kept_rows) >= WANTED_KEPT and len(discarded_rows) >= WANTED_DISCARDED:
            break

if not kept_rows and not discarded_rows:
    display(Markdown('**No changed rows found in scanned real data. Increase MAX_SCAN_LINES.**'))
else:
    df = pd.DataFrame(kept_rows + discarded_rows)[
        ['Year', 'Raw N-gram', 'Processed N-gram', 'Outcome', 'Result']
    ].sort_values('Year').reset_index(drop=True)

    display(Markdown('#### Real Changed Rows (Kept + Discarded)'))
    display(HTML(df.to_html(index=False, escape=False)))
    display(Markdown(
        f'Changed rows scanned before sampling stop: {changed_rows_scanned} | '
        f'Distinct kept shown: {len(kept_rows)} | Distinct discarded shown: {len(discarded_rows)}'
    ))

#### Real Changed Rows (Kept + Discarded)

Year,Raw N-gram,Processed N-gram,Outcome,Result
1901,accursèd be that tongue that,accursed be that tongue that,Kept,OK
1904,account of the portolá expedition,account of the portola expedition,Kept,OK
1905,accountant a certificate of registration,accountant certificate of registration,Discarded (<5 tokens),X
1907,achsah as a wife to,achsah as wife to,Discarded (<5 tokens),X
1916,achats industriels pour les régions,achats industriels pour les regions,Kept,OK
1936,account of lincoln s death,account of lincoln death,Discarded (<5 tokens),X
1947,accounting systems for contractors a,accounting systems for contractors,Discarded (<5 tokens),X
1965,account of shakespeare s english,account of shakespeare english,Discarded (<5 tokens),X
1973,accounting does not have a,accounting does not have,Discarded (<5 tokens),X
1976,accounting clerks l accounting clerks,accounting clerks accounting clerks,Discarded (<5 tokens),X


Changed rows scanned before sampling stop: 39806 | Distinct kept shown: 8 | Distinct discarded shown: 8

This table compares **real n-grams** before and after preprocessing.
It includes both rows **kept** by the pipeline and rows **discarded** when fewer than 5 tokens remain after cleaning.
`Result` highlights the status: <span style="color:#2e7d32;"><strong>OK</strong></span> for kept rows and <span style="color:#c62828;"><strong>X</strong></span> for discarded rows.

### Decade Aggregation

After preprocessing, cleaned n-grams are grouped by decade, and each decade is saved in its own text file.

- Each line contains one processed n-gram
- Lines are **replicated by frequency**: if an n-gram has total `match_count = 5` in the 1900s, it is written 5 times
- This is important because embeddings learn from **exposure frequency**: more frequent patterns are seen more often during training, so the model captures their semantic weight more realistically
- In short, replication preserves the corpus distribution and prevents flattening rare and common n-grams into the same importance

In [5]:
# Load and analyze preprocessed decade files
from pathlib import Path
from IPython.display import display

def format_number(num: int) -> str:
    """Format large numbers as readable (e.g., 1234567 -> 1.2M)."""
    if num >= 1_000_000:
        return f'{num/1_000_000:.1f}M'
    elif num >= 1_000:
        return f'{num/1_000:.0f}K'
    else:
        return str(num)

processed_dir = Path(DATA_PROCESSED_EXPANDED_DIR)
decade_stats = []

for decade_file in sorted(processed_dir.glob('*s.txt')):
    decade = decade_file.stem  # Extract "1900s" from "1900s.txt"
    
    # Skip auxiliary stats file from the table
    if decade == 'vocab_stats':
        continue
    
    # Count total lines and unique n-grams
    with open(decade_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    total_lines = len(lines)
    unique_ngrams = len(set(lines))
    
    # Sample unique n-grams
    sample_ngrams = list(set(lines))[:3]
    
    decade_stats.append({
        'Decade': decade,
        'Total Lines': format_number(total_lines),
        'Unique N-grams': format_number(unique_ngrams),
        'Sample N-grams': ' | '.join(sample_ngrams)
    })

df_decades = pd.DataFrame(decade_stats)

# Improve readability: hide index and widen Sample N-grams column
styled_df = (
    df_decades.style
    .hide(axis='index')
    .set_properties(subset=['Sample N-grams'], **{
        'min-width': '560px',
        'white-space': 'normal',
        'text-align': 'left'
    })
    .set_properties(subset=['Decade'], **{'min-width': '90px'})
)

display(styled_df)

Decade,Total Lines,Unique N-grams,Sample N-grams
1900s,16.0M,1.6M,his whereabouts was in this | gallery is situated in the | if hawks chase doves through
1910s,16.1M,1.6M,police department in the form | women and children in war | gallery is situated in the
1920s,14.6M,1.5M,the clerk explained that this | gallery is situated in the | on the species of pinnixa
1930s,13.4M,1.4M,the clerk explained that this | if no reasonable basis of | gallery is situated in the
1940s,13.9M,1.4M,the clerk explained that this | section of the american people | if hawks chase doves through
1950s,19.0M,1.9M,mexico city and found that | section of the american people | world war ii president franklin
1960s,26.4M,2.6M,the clerk explained that this | section of the american people | world war ii president franklin
1970s,30.8M,2.9M,mexico city and found that | the army reserve forces policy | world war ii president franklin
1980s,34.8M,3.2M,mexico city and found that | the army reserve forces policy | world war ii president franklin
1990s,41.1M,3.7M,mexico city and found that | the army reserve forces policy | women and children in war


The table above summarizes the output of preprocessing **by decade**:

- **Decade**: historical time slice (1900s to 2010s)
- **Total Lines**: total training lines after frequency replication (higher values mean more textual evidence for that decade)
- **Unique N-grams**: lexical diversity after cleaning and filtering
- **Sample N-grams**: examples of final cleaned 5-grams actually used downstream

Why decade aggregation is useful:

1. It preserves **temporal structure**, so each model is trained on language from a specific period.
2. It preserves **usage intensity** through replicated lines, so frequent patterns keep their statistical weight.
3. It reduces noise after preprocessing, giving cleaner and more stable inputs for later stages.
4. It makes decades directly comparable, which is essential for alignment and semantic drift analysis.

---

## Phase 3: Vocabulary Building

This section demonstrates the vocabulary construction process: aggregating word frequencies across all decades and selecting the top words for consistent word-to-index mapping.

### Vocabulary Creation

The shared vocabulary is built by:

1. Loading all preprocessed decade files and counting token frequencies across all decades
2. Selecting the top 50,001 most frequent words
3. Building a fixed word-to-index mapping where `<UNK>` (unknown token) always maps to index 0

This creates a single, consistent index space for all decades, enabling direct comparison of embeddings over time.

In [10]:
# Load vocabulary
from IPython.display import display

vocab_path = Path(get_vocab_path(processed_dir=DATA_PROCESSED_EXPANDED_DIR))

with open(vocab_path, 'r', encoding='utf-8') as f:
    vocab = json.load(f)

print(f"Loaded vocabulary: {len(vocab):,} words")

# Show a larger, readable sample of index -> token mappings
SAMPLE_ROWS = 15

idx_word_pairs = sorted(vocab.items(), key=lambda kv: kv[1])
rows = []
seen_indices = set()

# Always show UNK (index 0) if present
unk_word = next((w for w, i in vocab.items() if i == 0), None)
if unk_word is not None:
    rows.append({'Index': 0, 'Word': unk_word})
    seen_indices.add(0)

for word, idx in idx_word_pairs:
    if idx in seen_indices:
        continue
    rows.append({'Index': idx, 'Word': word})
    seen_indices.add(idx)
    if len(rows) >= SAMPLE_ROWS:
        break

df_vocab_examples = pd.DataFrame(rows)

# Escape angle brackets so special tokens like <UNK> are visible in HTML tables
df_vocab_examples['Word'] = df_vocab_examples['Word'].apply(
    lambda w: w.replace('<', '&lt;').replace('>', '&gt;') if isinstance(w, str) else w
)

styled_vocab_examples = (
    df_vocab_examples.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th.col_heading', 'props': 'text-align: center;'},
        {'selector': 'th.col0, td.col0', 'props': 'min-width: 90px;'},
        {'selector': 'th.col1, td.col1', 'props': 'min-width: 180px;'}
    ], overwrite=False)
)

display(styled_vocab_examples)

Loaded vocabulary: 50,002 words


Index,Word
0,<UNK>
1,the
2,of
3,that
4,to
5,and
6,on
7,is
8,in
9,his


### Vocabulary Examples

This table shows a small sample of the **final mapping used by the model**: each token is converted into a fixed integer ID.

- **Index**: numeric ID used in dataset tensors and in the embedding layer
- **Word**: token associated with that ID in the shared vocabulary

The table below shows 15 mappings (starting with index 0), and can verify that frequent words like "the", "of", "that" receive low indices.

In [7]:
# Analyze vocabulary coverage across decades
import re
from IPython.display import display

coverage_stats = []

for decade_file in sorted(processed_dir.glob('*s.txt')):
    decade = decade_file.stem  # Extract "1900s" from "1900s.txt"
    
    # Keep only real decades (e.g., 1900s, 1910s, ..., 2010s)
    if not re.fullmatch(r'\d{4}s', decade):
        continue
    
    # Load words from this decade
    decade_words = set()
    with open(decade_file, 'r', encoding='utf-8') as f:
        for line in f:
            ngram_text = line.strip()
            if ngram_text:
                for word in ngram_text.split():
                    decade_words.add(word)
    
    # Count how many are in the vocabulary
    vocab_words = [w for w in decade_words if w in vocab]
    coverage_pct = 100 * len(vocab_words) / len(decade_words) if decade_words else 0
    
    coverage_stats.append({
        'Decade': decade,
        'Vocab Coverage': f'{coverage_pct:.1f}%',
        'In Vocab': format_number(len(vocab_words)),
        'Unique Words': format_number(len(decade_words))
    })

df_coverage = pd.DataFrame(coverage_stats)

# Improve readability without using subset=... (avoids typing warnings in some checkers)
styled_coverage = (
    df_coverage.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th.col_heading', 'props': 'text-align: center;'},
        {'selector': 'th.col0, td.col0', 'props': 'min-width: 90px;'},
        {'selector': 'th.col1, td.col1', 'props': 'min-width: 130px;'},
        {'selector': 'th.col2, td.col2, th.col3, td.col3', 'props': 'min-width: 120px;'}
    ], overwrite=False)
)

display(styled_coverage)

Decade,Vocab Coverage,In Vocab,Unique Words
1900s,48.1%,38K,80K
1910s,53.3%,39K,73K
1920s,55.1%,40K,72K
1930s,59.6%,40K,67K
1940s,62.3%,41K,66K
1950s,53.5%,44K,82K
1960s,45.1%,47K,104K
1970s,43.4%,48K,111K
1980s,40.1%,49K,122K
1990s,34.2%,49K,145K


This table evaluates how well the vocabulary covers the words in each decade.

<strong>Column meanings</strong>:
- <span style="color:#2ca02c;"><strong>Decade</strong></span>: historical period (1900s to 2010s)
- <span style="color:#ff7f0e;"><strong>Unique Words</strong></span>: total distinct words in that decade
- <span style="color:#9467bd;"><strong>In Vocab</strong></span>: how many are in the shared vocabulary
- <span style="color:#d62728;"><strong>Vocab Coverage</strong></span>: percentage covered

The Vocab Coverage means:

- **Higher coverage** means the decade is **better represented** by the fixed vocabulary,
- **Lower coverage** means more words **fall outside the vocabulary** and will be mapped to `<UNK>`,

The gap between **Unique Words** and **In Vocab** tells you how much lexical information is compressed into `<UNK>`


#### Vocabulary summary
The vocabulary is now complete: **50,002 entries** with index 0 as `<UNK>` (fallback) and indices 1–50,001 for the most frequent words.
This fixed mapping ensures all decades use the same indices, enabling direct embedding comparison across time.
In the next phase, n-grams will be converted to index sequences using this vocabulary for CBOW model training.